# Calibration de `beta` (force conditionnelle du bracket)

On simule un grand nombre de tournois complets (poules + bracket) et on compte combien de fois chaque équipe est **championne**. On compare cette fréquence empirique à la probabilité **théorique** de titre = `cote_victoire` dé-viggée par **Shin** (le simulateur utilise désormais cette même proba Shin, ainsi que la qualification dé-viggée par logit-shift).

`beta` règle l'amortissement de la force conditionnelle `cv_true / surv**beta` :
- `beta = 1.0` : conditionnement complet (identité bayésienne exacte sous indépendance) ;
- `beta = 0.0` : aucun conditionnement (ratio brut des probas de victoire) ;
- `beta ∈ (0,1)` : conditionnement partiel (limite le boost des outsiders).

Le `beta` optimal est celui qui **minimise l'écart** entre fréquence empirique et théorique. **À re-calibrer** après le passage au dé-vig Shin/logit-shift (le `beta=0.95` historique a été calibré sur l'ancien modèle 1/cote brut).

In [1]:
import time
import numpy as np
import pandas as pd
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().parent
sys.path.append(str(PROJECT_DIR))
from mpp_project.bracket_simulator import simulate_champion_distribution
from mpp_project.core import calculate_true_outcome_probas_from_odds

DATA_DIR = PROJECT_DIR / "data"
df_odds = pd.read_csv(DATA_DIR / "CDM_2026_group_stage_odds.csv")

# Probabilité théorique de titre = cote_victoire dé-viggée par SHIN (somme = 1).
# IMPORTANT : le simulateur utilise désormais cette même proba Shin (dé-vig), donc la
# cible de calibration doit être Shin (et NON 1/cote brut normalisé), sinon le beta
# optimal compenserait à tort l'écart Shin <-> brut.
theo = np.asarray(calculate_true_outcome_probas_from_odds(df_odds["cote_victoire"].values), dtype=float)
theo = theo / theo.sum()

In [2]:
# ==========================================
# 1. SIMULATION POUR UN BETA DONNÉ (tableau détaillé)
# ==========================================
N_RUNS = 1_000_000   # ~24 min. Réduire à 100_000-200_000 (~2-5 min) pour itérer vite.
BETA = 0.95

print(f"Simulation de {N_RUNS:,} tournois (beta={BETA})...")
t0 = time.time()
counts, names = simulate_champion_distribution(df_odds, n_runs=N_RUNS, beta=BETA, verbose=True)
print(f"-> terminé en {time.time()-t0:.0f}s\n")

emp = counts / counts.sum()
res = pd.DataFrame({
    "equipe": names,
    "emp_%": emp * 100,
    "theo_%": theo * 100,
}).sort_values("theo_%", ascending=False).reset_index(drop=True)
res["ecart_pts"] = res["emp_%"] - res["theo_%"]

pd.set_option("display.max_rows", None)
print(res.to_string(index=False, float_format=lambda x: f"{x:6.2f}"))

mae = np.mean(np.abs(emp - theo)) * 100
maxe = np.max(np.abs(emp - theo)) * 100
print(f"\nÉcart absolu moyen : {mae:.3f} pts | écart max : {maxe:.3f} pts")
print(f"(beta={BETA} ; un écart faible = la simulation reproduit bien cote_victoire)")

Simulation de 1,000,000 tournois (beta=0.95)...


KeyboardInterrupt: 

In [5]:
# ==========================================
# 2. BALAYAGE DE BETA (pour trouver la meilleure calibration)
# ==========================================
# On compare l'ajustement (écart absolu moyen empirique vs théorique) pour plusieurs
# beta, à N réduit (la précision suffit pour classer les beta entre eux).
N_SWEEP = 100_000
BETAS = [0.88, 0.885, 0.89, 0.895, 0.9, 0.825, 0.85, 0.925, 0.95, 0.975]

print(f"Balayage de beta ({N_SWEEP:,} tournois chacun)...\n")
print(f"{'beta':>5} | {'MAE (pts)':>10} | {'max (pts)':>10}")
print("-" * 32)
best = None
for b in BETAS:
    c, _ = simulate_champion_distribution(df_odds, n_runs=N_SWEEP, beta=b, verbose=False, seed=0)
    e = c / c.sum()
    mae_b = np.mean(np.abs(e - theo)) * 100
    maxe_b = np.max(np.abs(e - theo)) * 100
    print(f"{b:5.2f} | {mae_b:10.3f} | {maxe_b:10.3f}")
    if best is None or mae_b < best[1]:
        best = (b, mae_b)

print(f"\n-> meilleur ajustement : beta = {best[0]} (MAE = {best[1]:.3f} pts)")
print("   (le beta qui minimise l'écart reproduit le mieux les cotes de victoire)")

Balayage de beta (100,000 tournois chacun)...

 beta |  MAE (pts) |  max (pts)
--------------------------------
 0.88 |      0.033 |      0.122
 0.89 |      0.032 |      0.119
 0.89 |      0.032 |      0.131
 0.90 |      0.031 |      0.183
 0.90 |      0.032 |      0.222


KeyboardInterrupt: 